# Character Constituency based rescoring

## Load Speechain model

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm
from types import SimpleNamespace

from speechain.utilbox.yaml_util import load_yaml
from speechain.utilbox.data_loading_util import read_data_by_path
from speechain.runner import Runner

tqdm.pandas()

In [3]:
SPEECHAIN_ROOT = "/home/is/r-ghimire/speechain"
EXP_DIR    = f"{SPEECHAIN_ROOT}/recipes/asr/slr54nepaliasr/exp"
CHECKPOINT = "models/10_valid_accuracy_average.pth"
DEVICE = "cuda:1"

In [4]:
def load_asr_model(model_name):
    os.environ.setdefault("SPEECHAIN_ROOT", SPEECHAIN_ROOT)

    exp_cfg = load_yaml(f"{EXP_DIR}/{model_name}/exp_cfg.yaml")
    # print(exp_cfg)
    model_cfg = exp_cfg["train_cfg"]["model"]

    args = SimpleNamespace(
        train_result_path=f"{EXP_DIR}/{model_name}",
        non_blocking=True,
        distributed=False,
    )
    device = torch.device(DEVICE)
    model = Runner.build_model(model_cfg, args=args, device=device)
    ckpt_path = f"{EXP_DIR}/{model_name}/{CHECKPOINT}"
    print(f"Loading model from : {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt["latest_model"] if "latest_model" in ckpt else ckpt
    model.load_state_dict(state, strict=True)
    model.eval()
    return model

model_large_ccnn = load_asr_model("ne_char_conformer-large-v2_lr2e-3-ccnn-only")
model_large = load_asr_model("ne_char_conformer-large-v2_lr2e-3")

Loading model from : /home/is/r-ghimire/speechain/recipes/asr/slr54nepaliasr/exp/ne_char_conformer-large-v2_lr2e-3-ccnn-only/models/10_valid_accuracy_average.pth
Loading model from : /home/is/r-ghimire/speechain/recipes/asr/slr54nepaliasr/exp/ne_char_conformer-large-v2_lr2e-3/models/10_valid_accuracy_average.pth


Get the top-k transcript from Conformer

In [5]:
def transcribe(audio_file_name, model, infer_conf={}):
    wav, sr = read_data_by_path(audio_file_name, return_tensor=True, return_sample_rate=True)
    if sr != 16000:
        wav = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(wav.squeeze(-1)).unsqueeze(-1)

    feat = wav.unsqueeze(0).to(DEVICE)  # (B,T,1)
    feat_len = torch.tensor([wav.shape[0]], device=DEVICE)
    infer_conf = {
        "beam_size": 30,
        "ctc_weight": 0.2,
        "decode_only": True,
        "sent_per_beam" : 10,
        "temperature": 1.6,
    }
    with torch.inference_mode():
        out = model.inference(
            infer_conf=infer_conf,
            feat=feat,
            feat_len=feat_len,
            decode_only=True,
        )
    return out

In [6]:
test_folder = "/home/is/r-ghimire/speechain/datasets/slr54nepaliasr/data/wav16000/test"
with open(os.path.join(test_folder, "idx2text"), encoding = "utf-8") as f:
        id2text = dict(line.rstrip("\n").split(" ", 1) for line in f)
with open(os.path.join(test_folder, "idx2wav"), encoding = "utf-8") as f:
    id2wav = dict(line.rstrip("\n").split(" ", 1) for line in f)

df = (
    pd.DataFrame({
        "text": pd.Series(id2text),
        "wav_path": pd.Series(id2wav),
    })
    .rename_axis("id")
    .reset_index()
)
df.head(5)

,id,text,wav_path
0,00033e7a6c,धेरै धन्यवाद,/home/is/r-ghimire/speechain/datasets/slr54nep...
1,000414cbb9,थप आकर्षण हुन्,/home/is/r-ghimire/speechain/datasets/slr54nep...
2,00062958ef,भाग लिन थाले,/home/is/r-ghimire/speechain/datasets/slr54nep...
3,00097747cd,गिँडेर ममता,/home/is/r-ghimire/speechain/datasets/slr54nep...
4,00180ecd4a,विकी सायद त्यै,/home/is/r-ghimire/speechain/datasets/slr54nep...


In [7]:
for idx, row in df[0:1].iterrows():
    fileid = row["id"]
    ref_text = row["text"]
    path = row["wav_path"]

    transcript = transcribe(path, model_large)
    gen_large = transcript['text']['content'][0]

    transcript = transcribe(path, model_large_ccnn)
    gen_large_ccnn = transcript['text']['content'][0]

    print(f"{fileid} Ref: {ref_text}  Gen(large): {gen_large}   Gen(large_ccnn): {gen_large_ccnn}")

00033e7a6c Ref: धेरै धन्यवाद  Gen(large): ['धेरै धन्यवाद', 'धेरै धन्यावाद', 'धेरै धन्यवादर', 'धेरै धन्यवादद', 'धेरै धन्यवबाद', 'धेरै धन्यवाद्', 'धेरै धधन्यवाद', 'धेरै धन्यववाद', 'धेरै धन्यवााद', 'धेरै धन्यबाद']   Gen(large_ccnn): ['धेरै धन्यवाद', 'धेरै धन्यवादद', 'धेरै धन्यावाद', 'धेरै धधन्यवाद', 'धधेरै धन्यवाद', 'धेरै धन्यवद', 'धेरै धन्यवादर', 'धेरै धन्पवाद', 'धेरै धान्यवाद', 'धेरै धनव्यवाद']


In [8]:
transcript

{'text': {'format': 'txt',
  'content': [['धेरै धन्यवाद',
    'धेरै धन्यवादद',
    'धेरै धन्यावाद',
    'धेरै धधन्यवाद',
    'धधेरै धन्यवाद',
    'धेरै धन्यवद',
    'धेरै धन्यवादर',
    'धेरै धन्पवाद',
    'धेरै धान्यवाद',
    'धेरै धनव्यवाद']]},
 'text_len': {'format': 'txt',
  'content': [[13, 14, 14, 14, 14, 12, 14, 13, 14, 14]]},
 'feat_token_len_ratio': {'format': 'txt',
  'content': [[8.083333015441895,
    7.461538314819336,
    7.461538314819336,
    7.461538314819336,
    7.461538314819336,
    8.818181991577148,
    7.461538314819336,
    8.083333015441895,
    7.461538314819336,
    7.461538314819336]]},
 'text_confid': {'format': 'txt',
  'content': [[-0.2977210581302643,
    -0.5621840953826904,
    -0.7081677913665771,
    -0.7135331630706787,
    -0.7251191735267639,
    -0.7290472388267517,
    -0.7338499426841736,
    -0.7364158034324646,
    -0.7424778342247009,
    -0.7571907043457031]]}}

## Load CCNN

In [9]:
# !pip install -e /home/is/r-ghimire/decoding

In [10]:
import torch
import torch.nn.functional as F
from CCNN.config import get_config
from CCNN.vocab import Vocab
from CCNN.model import CCNN
from CCNN.decode import beam_search_ccnn, score_next_with_ccnn
from CCNN.rules import is_valid_next

config = get_config()
# config

Load CCNN Vocab

In [11]:
CCNN_VOCAB_PATH = "/home/is/r-ghimire/decoding/CCNN/ne_speechain.vocab"
CCNN_CHECKPOINT = "/home/is/r-ghimire/decoding/checkpoint/ccnn_lstm_slr54_ep50_new_tokenizer.pt"

In [12]:
with open(CCNN_VOCAB_PATH, "r", encoding="utf-8") as vocabfile:
    tokens = [v.rstrip('\n') for v in vocabfile]
vocab = Vocab(tokens=tokens)
len(vocab.tokens)

66

In [13]:
DEVICE

'cuda:1'

In [14]:
ccnn_model = CCNN(
    vocab_size  = len(vocab.tokens),
    pad_id      = vocab.pad_id,
    d_model     = config.d_model,
    n_layers    = config.layers,
    dropout     = config.dropout,
)
ccnn_ckpt = torch.load(CCNN_CHECKPOINT, map_location=DEVICE, weights_only=False)

ccnn_model.load_state_dict(ccnn_ckpt["model"])
ccnn_model.to(DEVICE)
ccnn_model.eval()
ccnn_model

CCNN(
  (emb): Embedding(66, 256, padding_idx=0)
  (dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(256, 256, num_layers=2, batch_first=True, dropout=0.3)
  (lm_head): Linear(in_features=256, out_features=66, bias=True)
)

## Rescore Function

In [15]:
def get_CCNN_Score(ccnn_model, imput_tokens, use_ccnn = False):

    if isinstance(imput_tokens, torch.Tensor):
        tokens = imput_tokens.detach().cpu().tolist()
    else:
        tokens = list(imput_tokens)

    # will calculate LM and cond score and return both
    ccnn_model.eval()

    # Need at least 2 tokens to score next-token prediction
    if len(tokens) < 2:
        return 0.0, 0.0

    # Teacher-forcing setup:
    # x predicts next token target
    x = tokens[:-1]
    y = tokens[1:]
    T = len(x)

    x_tensor = torch.tensor(x, dtype=torch.long, device=DEVICE).unsqueeze(0)   # (1,T)
    y_tensor = torch.tensor(y, dtype=torch.long, device=DEVICE)               # (T,)
    lengths = torch.tensor([T], dtype=torch.long, device=DEVICE)              # (1,)

    # cand_ids is the "correct next token" as K=1 candidate at each time step
    cand_ids = y_tensor.view(1, T, 1)                                         # (1,T,1)

    with torch.no_grad():
        lm_logits, cons_logits = ccnn_model(x_tensor, lengths, cand_ids)      # (1,T,V), (1,T,1)

        # LM score: sum_t log p(y_t | x_<=t)
        logp = F.log_softmax(lm_logits[0], dim=-1)                            # (T,V)
        lm_score = logp[torch.arange(T, device=DEVICE), y_tensor].sum().item()

        # Constraint score: sum_t log sigmoid(cons_logits_t)
        if use_ccnn:
            cons_score = F.logsigmoid(cons_logits[0, :, 0]).sum().item()
        else:
            cons_score = 0.0

    return lm_score, cons_score, T

In [16]:
# transcript.keys()
# transcript["text"]["content"][0]

Hyperparameters

In [17]:
ALPHA = 0.05 # LM weight
BETA  = 0.02 # CONS weight
DELTA = 1.0 # Length normalization factor

In [18]:
def norm_sum(s, L, delta=1.0):
    return float(s) / (float(L) ** delta)


def rescore(transcript):
    best_am_score = -1e18
    best_hyp_am = None
    best_hyp = None
    best_total = -1e18

    hyp_strings = transcript["text"]["content"][0]
    am_scores   = transcript["text_confid"]["content"][0]

    for hyp_string, am_score in zip(hyp_strings, am_scores):
        am_score   = float(am_score)
        hyp_tokens = vocab.encode_atomic(hyp_string)
        # print(f"hyp_string = {hyp_string} = {am_score}")
        lm_score, cons_score, T = get_CCNN_Score(ccnn_model, hyp_tokens, use_ccnn=True)
        L = max(T, 1)
        lm_norm = norm_sum(lm_score, L, DELTA)
        cons_norm = norm_sum(cons_score, L, DELTA)

        total = am_score + ALPHA * lm_norm + BETA * cons_norm

        if total > best_total:
            best_total = total
            best_hyp = hyp_string
            scores = { "am": am_score, "lm": lm_norm, "cons": cons_norm }
        if am_score > best_am_score:
            best_am_score = am_score
            best_hyp_am = hyp_string

    return {
        "score":scores,
        "text":{
            "am" :best_hyp_am,
            "total": best_hyp
        }
    }

rescore(transcript)

In [19]:
rescore(transcript)

{'score': {'am': -0.2977210581302643,
  'lm': -1.9083087627704327,
  'cons': -8.35927675797603e-09},
 'text': {'am': 'धेरै धन्यवाद', 'total': 'धेरै धन्यवाद'}}

## Rescore over test dataset


In [20]:
def read_test_ds(test_ds):
    with open(os.path.join(test_ds, "idx2text"), encoding = "utf-8") as f:
            id2text = dict(line.rstrip("\n").split(" ", 1) for line in f)
    with open(os.path.join(test_ds, "idx2wav"), encoding = "utf-8") as f:
        id2wav = dict(line.rstrip("\n").split(" ", 1) for line in f)

    df = (
        pd.DataFrame({
            "text": pd.Series(id2text),
            "wav_path": pd.Series(id2wav),
        })
        .rename_axis("id")
        .reset_index()
    )
    return df

In [21]:
def generate_transcripts(test_ds_path, out_tsv_path, N):
    rows = []

    df = read_test_ds(test_ds_path)
    df = df.head(N).reset_index(drop=True)

    for idx, row in df.iterrows():
        fileid = row["id"]
        ref_text = row["text"]
        path = row["wav_path"]

        trans_large = transcribe(path, model_large)
        trans_large_ccnn = transcribe(path, model_large_ccnn)

        res_large = rescore(trans_large)
        res_ccnn  = rescore(trans_large_ccnn)

        r = {
            "id": fileid,
            "wav_path": path,
            "ref": ref_text,

            # Large (baseline)
            "hyp_large_am":    res_large["text"]["am"],
            "hyp_large_total": res_large["text"]["total"],
            "am_large":        res_large["score"]["am"],
            "lm_large":        res_large["score"]["lm"],
            "cons_large":      res_large["score"]["cons"],

            # Large + CCNN integrated (or whatever model_large_ccnn is)
            "hyp_ccnn_am":     res_ccnn["text"]["am"],
            "hyp_ccnn_total":  res_ccnn["text"]["total"],
            "am_ccnn":         res_ccnn["score"]["am"],
            "lm_ccnn":         res_ccnn["score"]["lm"],
            "cons_ccnn":       res_ccnn["score"]["cons"],
        }
        rows.append(r)

        # optional progress print
        if (idx + 1) % 50 == 0:
            print(f"DONE == {idx+1}/{len(df)}")

    out_df = pd.DataFrame(rows)
    # Save as TSV
    out_df.to_csv(out_tsv_path, sep="\t", index=False)
    print("saved:", out_tsv_path)


In [22]:
# Run
test_ds_path = "/home/is/r-ghimire/speechain/datasets/slr54nepaliasr/data/wav16000/test"
out_tsv_path = "/home/is/r-ghimire/speechain/dump/rescore_1000.tsv"
N = 1000
generate_transcripts(test_ds_path, out_tsv_path, N)

DONE == 50/1000
DONE == 100/1000
DONE == 150/1000
DONE == 200/1000
DONE == 250/1000
DONE == 300/1000
DONE == 350/1000
DONE == 400/1000
DONE == 450/1000
DONE == 500/1000
DONE == 550/1000
DONE == 600/1000
DONE == 650/1000
DONE == 700/1000
DONE == 750/1000
DONE == 800/1000
DONE == 850/1000
DONE == 900/1000
DONE == 950/1000
DONE == 1000/1000
saved: /home/is/r-ghimire/speechain/dump/rescore_1000.tsv


In [23]:
# Read
out_tsv_path = "/home/is/r-ghimire/speechain/dump/rescore_1000.tsv"
out_df = pd.read_csv(out_tsv_path, sep='\t')

In [24]:
out_df.head(10)

,id,wav_path,ref,hyp_large_am,hyp_large_total,am_large,lm_large,cons_large,hyp_ccnn_am,hyp_ccnn_total,am_ccnn,lm_ccnn,cons_ccnn
0,00033e7a6c,/home/is/r-ghimire/speechain/datasets/slr54nep...,धेरै धन्यवाद,धेरै धन्यवाद,धेरै धन्यवाद,-0.301780,-1.908309,-8.359277e-09,धेरै धन्यवाद,धेरै धन्यवाद,-0.297721,-1.908309,-8.359277e-09
1,000414cbb9,/home/is/r-ghimire/speechain/datasets/slr54nep...,थप आकर्षण हुन्,थप आकर्षण हुन्,थप आकर्षण हुन्,-0.294788,-2.265966,-1.043558e-08,थप आकर्षण हुन्,थप आकर्षण हुन्,-0.277493,-2.265966,-1.043558e-08
2,00062958ef,/home/is/r-ghimire/speechain/datasets/slr54nep...,भाग लिन थाले,भाग लिन थाले,भाग लिन थाले,-0.315674,-2.121014,-1.298673e-07,भाग लिन थाले,भाग लिन थाले,-0.306461,-2.121014,-1.298673e-07
3,00097747cd,/home/is/r-ghimire/speechain/datasets/slr54nep...,गिँडेर ममता,घिँडे र ममता,घिँडे र ममता,-0.374765,-2.654073,-1.775510e+00,घिँडेर ममता,घिँडेर ममता,-0.368761,-2.790859,-1.923470e+00
4,00180ecd4a,/home/is/r-ghimire/speechain/datasets/slr54nep...,विकी सायद त्यै,विकि साय,विकि साय,-0.326398,-2.623604,-2.777158e-07,विकि साय,विकि साय,-0.309861,-2.623604,-2.777158e-07
5,001aafcee1,/home/is/r-ghimire/speechain/datasets/slr54nep...,भारतको राष्ट्रिय हिन्दी,भारतको राष्ट्रिय हिन्दी,भारतको राष्ट्रिय हिन्दी,-0.293479,-1.222749,-5.001419e-07,भारतको राष्ट्रिय हिन्दी,भारतको राष्ट्रिय हिन्दी,-0.263712,-1.222749,-5.001419e-07
6,001ca0ce21,/home/is/r-ghimire/speechain/datasets/slr54nep...,बसेको हुन्छ,बसेको हुन्छ,बसेको हुन्छ,-0.328768,-1.280699,-2.588165e-08,बसेको हुन्छ,बसेको हुन्छ,-0.292608,-1.280699,-2.588165e-08
7,001d884c4a,/home/is/r-ghimire/speechain/datasets/slr54nep...,अनि तिनका अर्थहरूका,अनि तिनका अर्थहरूका,अनि तिनका अर्थहरूका,-0.318709,-1.692447,-7.777256e-08,अनि तिनका अर्थहरूका,अनि तिनका अर्थहरूका,-0.325054,-1.692447,-7.777256e-08
8,0021bc4fcb,/home/is/r-ghimire/speechain/datasets/slr54nep...,आमा री वन,आमा रिवन,आमा रिवन,-0.365865,-2.814392,-1.715563e-07,आमा रिवन,आमा रिवन,-0.356480,-2.814392,-1.715563e-07
9,0023f1da75,/home/is/r-ghimire/speechain/datasets/slr54nep...,विस्तारित गरिए पछि,विस्तारित गरिए पछि,विस्तारित गरिए पछि,-0.308087,-1.636618,-7.684397e-08,विस्तारित गरिए पछि,विस्तारित गरिए पछि,-0.302193,-1.636618,-7.684397e-08


## Evaluation of effectiveness of CCNN in Rescoring

In [25]:
import pandas as pd

# --- edit distance helpers ---
def _edit_distance(a, b):
    # a and b are lists (tokens)
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            cur = dp[j]
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[j] = min(
                dp[j] + 1,      # deletion
                dp[j - 1] + 1,  # insertion
                prev + cost     # substitution
            )
            prev = cur
    return dp[m]

def _wer(ref, hyp):
    ref_w = str(ref).strip().split()
    hyp_w = str(hyp).strip().split()
    if len(ref_w) == 0:
        return 0.0 if len(hyp_w) == 0 else 1.0
    return _edit_distance(ref_w, hyp_w) / len(ref_w)

def _cer(ref, hyp):
    ref_c = list(str(ref).strip())
    hyp_c = list(str(hyp).strip())
    if len(ref_c) == 0:
        return 0.0 if len(hyp_c) == 0 else 1.0
    return _edit_distance(ref_c, hyp_c) / len(ref_c)

def evaluate(tsv_path):
    df = pd.read_csv(tsv_path, sep="\t")
    # Required columns
    cols = [
        "ref",
        "hyp_large_am", "hyp_large_total",
        "hyp_ccnn_am",  "hyp_ccnn_total",
    ]
    for c in cols:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    # Compute per-utt WER/CER
    for name in ["large_am", "large_total", "ccnn_am", "ccnn_total"]:
        hyp_col = f"hyp_{name}"
        df[f"wer_{name}"] = [
            _wer(r, h) for r, h in zip(df["ref"], df[hyp_col])
        ]
        df[f"cer_{name}"] = [
            _cer(r, h) for r, h in zip(df["ref"], df[hyp_col])
        ]

    # Aggregate
    def _mean(x): return float(df[x].mean())

    summary = {
        "N": len(df),
        "WER_large_am": _mean("wer_large_am"),
        "WER_large_total": _mean("wer_large_total"),
        "WER_ccnn_am": _mean("wer_ccnn_am"),
        "WER_ccnn_total": _mean("wer_ccnn_total"),
        "CER_large_am": _mean("cer_large_am"),
        "CER_large_total": _mean("cer_large_total"),
        "CER_ccnn_am": _mean("cer_ccnn_am"),
        "CER_ccnn_total": _mean("cer_ccnn_total"),
    }

    # Relative improvements (negative is worse)
    summary["WER_gain_large_rescore_abs"] = summary["WER_large_am"] - summary["WER_large_total"]
    summary["WER_gain_ccnn_rescore_abs"]  = summary["WER_ccnn_am"]  - summary["WER_ccnn_total"]
    summary["CER_gain_large_rescore_abs"] = summary["CER_large_am"] - summary["CER_large_total"]
    summary["CER_gain_ccnn_rescore_abs"]  = summary["CER_ccnn_am"]  - summary["CER_ccnn_total"]

    # Paired win/loss for rescoring
    def paired_counts(base, new):
        d = df[base] - df[new]  # positive means new is better (lower error)
        return {
            "improved": int((d > 0).sum()),
            "worsened": int((d < 0).sum()),
            "unchanged": int((d == 0).sum()),
            "mean_delta": float(d.mean()),
        }

    paired = {
        "WER_large_am_vs_total": paired_counts("wer_large_am", "wer_large_total"),
        "WER_ccnn_am_vs_total":  paired_counts("wer_ccnn_am",  "wer_ccnn_total"),
        "CER_large_am_vs_total": paired_counts("cer_large_am", "cer_large_total"),
        "CER_ccnn_am_vs_total":  paired_counts("cer_ccnn_am",  "cer_ccnn_total"),
    }

    # Show best/worst changes (inspect failures)
    df["wer_delta_ccnn_rescore"] = df["wer_ccnn_am"] - df["wer_ccnn_total"]
    best = df.sort_values("wer_delta_ccnn_rescore", ascending=False).head(20)[
        ["id", "ref", "hyp_ccnn_am", "hyp_ccnn_total", "wer_ccnn_am", "wer_ccnn_total", "wer_delta_ccnn_rescore"]
    ]
    worst = df.sort_values("wer_delta_ccnn_rescore", ascending=True).head(20)[
        ["id", "ref", "hyp_ccnn_am", "hyp_ccnn_total", "wer_ccnn_am", "wer_ccnn_total", "wer_delta_ccnn_rescore"]
    ]

    return df, summary, paired, best, worst


In [26]:
from pprint import pprint

# Example usage:
df_eval, summary, paired, best, worst = evaluate(out_tsv_path)

pprint(summary, indent=4)
pprint(paired, indent=4)

# print(best.to_string(index=False))
# print(worst.to_string(index=False))

{   'CER_ccnn_am': 0.09494513768020933,
    'CER_ccnn_total': 0.09467467970555304,
    'CER_gain_ccnn_rescore_abs': 0.00027045797465628796,
    'CER_gain_large_rescore_abs': 0.00012652260020681871,
    'CER_large_am': 0.09620113421550795,
    'CER_large_total': 0.09607461161530113,
    'N': 1000,
    'WER_ccnn_am': 0.22735112665112667,
    'WER_ccnn_total': 0.22620112665112663,
    'WER_gain_ccnn_rescore_abs': 0.0011500000000000399,
    'WER_gain_large_rescore_abs': 0.00016666666666667607,
    'WER_large_am': 0.2336427933177933,
    'WER_large_total': 0.23347612665112663}
{   'CER_ccnn_am_vs_total': {   'improved': 9,
                                'mean_delta': 0.0002704579746562934,
                                'unchanged': 983,
                                'worsened': 8},
    'CER_large_am_vs_total': {   'improved': 6,
                                 'mean_delta': 0.00012652260020681077,
                                 'unchanged': 989,
                                 'wor